In [4]:
import pandas as pd
import yfinance as yf
import requests


In [5]:
headers = {"User-Agent": "Mozilla/5.0"}
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

tables = pd.read_html(requests.get(url, headers=headers).content)
tickers = tables[0]["Symbol"].str.replace(".", "-", regex=False).tolist()
print(f"Found {len(tickers)} tickers.")
tickers[:10]

Found 503 tickers.


['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A']

In [ ]:
PERIOD = '1y'

raw = {}
for i, ticker in enumerate(tickers):
    try:
        df = yf.Ticker(ticker).history(period=PERIOD, auto_adjust=True)
        if df.empty:
            continue
        df = df[['Open', 'High', 'Low', 'Close', 'Volume']]
        df.columns = ['open', 'high', 'low', 'close', 'volume']
        df.index = pd.to_datetime(df.index).tz_localize(None).normalize()
        df.index.name = 'date'
        raw[ticker] = df
    except Exception as e:
        print(f'Error {ticker}: {e}')
    if (i + 1) % 50 == 0:
        print(f'{i+1}/{len(tickers)} done...')

print(f'\nDownloaded: {len(raw)} tickers')

50/503 done...
100/503 done...
150/503 done...
200/503 done...
250/503 done...
300/503 done...
350/503 done...
400/503 done...
450/503 done...
500/503 done...

Downloaded: 503 tickers


In [ ]:
def clean(df):
    df = df[~df.index.duplicated(keep='last')]
    df = df.dropna(subset=['open', 'close'])
    df = df[(df['open'] > 0) & (df['close'] > 0) & (df['volume'] > 0)]
    return df

raw = {ticker: clean(df) for ticker, df in raw.items()}
print('cleaning done.')

Cleaning done.


In [ ]:
def passes_liquidity(df, min_dv=10_000_000):
    avg_dv = (df['close'] * df['volume']).tail(20).mean()
    return avg_dv >= min_dv

universe = {t: df for t, df in raw.items() if passes_liquidity(df)}
print(f'{len(universe)} tickers')

Universe after liquidity filter: 503 tickers


In [ ]:
summary = []
for ticker, df in universe.items():
    if len(df) < 130:
        continue
    r = (df['open'] / df['close'].shift(1)) - 1
    summary.append({
        'ticker': ticker,
        'avg_overnight_%': round(r.mean() * 100, 4),
        'std_%': round(r.std() * 100, 4),
        'rows': len(df)
    })

summary_df = pd.DataFrame(summary).sort_values('avg_overnight_%', ascending=False)
summary_df.head(10)

,ticker,avg_overnight_%,std_%,rows
406,SNDK,0.5592,3.3298,251
490,WDC,0.4099,2.1497,251
314,MU,0.3884,2.4025,251
387,Q,0.3661,1.4941,80
207,FCX,0.3487,2.0879,251
409,STX,0.3485,1.9235,251
119,FIX,0.3285,2.2684,251
439,TER,0.2954,2.4492,251
73,AVGO,0.2809,2.1344,251
212,GEV,0.2720,2.0115,251
